In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

In [ ]:
DATA_DIR = '../../../data/churn'

features = pd.read_csv(f'{DATA_DIR}/ecommerce_customer_features.csv')
targets = pd.read_csv(f'{DATA_DIR}/ecommerce_customer_targets.csv')

df = features.merge(targets, on='Customer_ID')
df['churned'] = df['churned'].map({'Yes': 1, 'No': 0})

df.head()

In [ ]:
FEATURES = [
    'account_age_months',
    'avg_order_value',
    'total_orders',
    'days_since_last_purchase',
    'discount_usage_rate',
    'return_rate',
    'browsing_frequency_per_week',
    'cart_abandonment_rate',
]
TARGET = 'churned'

X = df[FEATURES]
y = df[TARGET]

print(X.shape)
print(y.value_counts(normalize=True))

In [ ]:
# 클래스 불균형 확인
y.value_counts().plot(kind='bar', title='Churn Distribution')
plt.xticks([0, 1], ['Retained', 'Churned'], rotation=0)
plt.show()

In [ ]:
# 피처 상관관계
plt.figure(figsize=(10, 8))
sns.heatmap(df[FEATURES + [TARGET]].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation')
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression ===')
print(f'Accuracy : {accuracy_score(y_test, lr_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, lr_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, lr_proba):.4f}')

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(f'Accuracy : {accuracy_score(y_test, rf_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, rf_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, rf_proba):.4f}')

In [ ]:
# 피처 중요도 (Random Forest)
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances.plot(kind='bar', title='Feature Importances')
plt.tight_layout()
plt.show()